# Week 1

## Linear Algebra

### Task 1

![Image_2026-06-14_18-48-10.png](Image_2026-06-14_18-48-10.png)

### Task 2

![Image_2026-06-14_18-49-21.png](Image_2026-06-14_18-49-21.png)
![Image_2026-06-14_18-49-29.png](Image_2026-06-14_18-49-29.png)
![Image_2026-06-14_18-49-35.png](Image_2026-06-14_18-49-35.png)

### Task 3

![Image_2026-06-14_18-50-38.png](Image_2026-06-14_18-50-38.png)

### Task 4

![Image_2026-06-14_18-50-51.png](Image_2026-06-14_18-50-51.png)

## Python

### Task 1

In [2]:
import json
import time
import urllib.request
import urllib.error

STORE_ID = "DSAA31"
API_URL = (
    "https://ecapi.pchome.com.tw/ecshop/prodapi/v2/store/"
    f"{STORE_ID}/prod"
    "&offset={offset}&limit={limit}&fields=Id,Name,Price&_callback=jsonp_cb"
)
LIMIT = 36
OFFSET_START = 1
MAX_PAGES = 200
SLEEP_SEC = 0.3
OUTPUT_FILE = "products.txt"
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"),
    "Referer": f"https://24h.pchome.com.tw/store/{STORE_ID}",
}


def fetch(url):
    req = urllib.request.Request(url, headers=HEADERS)
    with urllib.request.urlopen(req, timeout=15) as resp:
        return resp.read().decode("utf-8", errors="replace")


def extract_json(text):
    text = text.strip()
    if not text.startswith(("{", "[")):
        lp = text.find("(")
        if lp != -1:
            text = text[lp + 1:]
    obj, _ = json.JSONDecoder().raw_decode(text.lstrip())
    return obj


def parse_ids(data):
    if isinstance(data, list):
        rows = data
    elif isinstance(data, dict):
        rows = data.get("Rows") or next((v for v in data.values() if isinstance(v, list)), [])
    else:
        rows = []
    return [item["Id"] for item in rows if isinstance(item, dict) and "Id" in item]


def crawl_all():
    seen = set()
    all_ids = []
    offset = OFFSET_START

    for page in range(1, MAX_PAGES + 1):
        url = API_URL.format(offset=offset, limit=LIMIT)
        data = extract_json(fetch(url))

        ids = parse_ids(data)
        if not ids:
            print(f"[第 {page} 頁] 無資料(offset={offset})，爬取結束。")
            break

        for pid in ids:
            if pid not in seen:
                seen.add(pid)
                all_ids.append(pid)
        print(f"[第 {page} 頁] offset={offset} 取回 {len(ids)} 筆，累計 {len(all_ids)} 筆。")

        offset += len(ids)
        time.sleep(SLEEP_SEC)

    return all_ids


def main():
    ids = crawl_all()
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for pid in ids:
            f.write(pid + "\n")
    for pid in ids:
        pass
    print(f"\n完成！共 {len(ids)} 個商品 ID，已寫入 {OUTPUT_FILE}")


if __name__ == "__main__":
    main()


[第 1 頁] offset=1 取回 36 筆，累計 36 筆。
[第 2 頁] offset=37 取回 10 筆，累計 46 筆。
[第 3 頁] 無資料(offset=47)，爬取結束。

完成！共 46 個商品 ID，已寫入 products.txt


### Task 2

In [3]:
import time

PROD_API = ("https://ecapi.pchome.com.tw/ecshop/prodapi/v2/prod/"
            "{pid}&fields=Id,ReviewCount,RatingValue&_callback=jsonp_cb")
SRC_FILE = "products.txt"
OUTPUT_FILE = "best-products.txt"
MIN_REVIEWS = 1
MIN_RATING = 4.9
SLEEP_SEC = 0.2


def load_product_ids(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def get_review_summary(pid):
    data = extract_json(fetch(PROD_API.format(pid=pid)))   # 重用 Task 1 的 fetch / extract_json
    info = data.get(pid, {}) if isinstance(data, dict) else {}
    return info.get("ReviewCount", 0) or 0, info.get("RatingValue")


def find_best_products(ids):
    best = []
    for pid in ids:
        count, rating = get_review_summary(pid)
        if count >= MIN_REVIEWS and rating is not None and rating > MIN_RATING:
            best.append(pid)
            print(f"  ✓ {pid}  評論 {count} 則，平均 {rating}")
        time.sleep(SLEEP_SEC)
    return best


def main():
    ids = load_product_ids(SRC_FILE)
    print(f"讀入 {len(ids)} 個商品 …")
    best = find_best_products(ids)

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for pid in best:
            f.write(pid + "\n")
    print(f"\n=== 至少 {MIN_REVIEWS} 則評論且平均評分 > {MIN_RATING} 的商品（共 {len(best)} 筆）===")
    for pid in best:
        print(pid)
    print(f"\n寫入 {OUTPUT_FILE}")


if __name__ == "__main__":
    main()


讀入 46 個商品 …
  ✓ DSAA31-A900JQR50-000  評論 1 則，平均 5
  ✓ DSAA31-1900JZ4LO-000  評論 4 則，平均 5
  ✓ DSAA31-A900IFZBC-000  評論 1 則，平均 5
  ✓ DSAA31-A900IFZEN-000  評論 1 則，平均 5
  ✓ DSAA31-A900J1950-000  評論 1 則，平均 5

=== 至少 1 則評論且平均評分 > 4.9 的商品（共 5 筆）===
DSAA31-A900JQR50-000
DSAA31-1900JZ4LO-000
DSAA31-A900IFZBC-000
DSAA31-A900IFZEN-000
DSAA31-A900J1950-000

寫入 best-products.txt


### Task 3

In [4]:
import re
import time

PROD_API = ("https://ecapi.pchome.com.tw/ecshop/prodapi/v2/prod/"
            "{pid}&fields=Id,Name,Price&_callback=jsonp_cb")
SRC_FILE = "products.txt"
SLEEP_SEC = 0.2


def load_product_ids(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def get_name_price(pid):
    data = extract_json(fetch(PROD_API.format(pid=pid)))
    info = data.get(pid, {}) if isinstance(data, dict) else {}
    price = (info.get("Price") or {}).get("P")
    return info.get("Name", ""), price


def is_intel_i5(name):
    return re.search(r"i5", name, re.IGNORECASE) is not None


def main():
    ids = load_product_ids(SRC_FILE)
    prices = []
    for pid in ids:
        name, price = get_name_price(pid)
        if price and is_intel_i5(name):
            prices.append(price)
            print(f"  i5 ▸ {price:>6}　{name}")
        time.sleep(SLEEP_SEC)

    if prices:
        avg = sum(prices) / len(prices)
        print(f"\nIntel i5 ASUS PC 共 {len(prices)} 筆，總價 {sum(prices)}")
        print(f"平均價格 = {avg:.1f} 元")
    else:
        print("找不到含 Intel i5 處理器的商品。")


if __name__ == "__main__":
    main()


  i5 ▸  33990　華碩 H-S503MER-514500076W(i5-14500/32G/1TB SSD/W11)
  i5 ▸  29888　華碩 H-S500TER-514500005W(i5-14500/16G/1TB SSD/W11)
  i5 ▸  24888　華碩 H-V500MV-13420H162W(i5-13420H/16G/1TB SSD/W11)
  i5 ▸  29888　華碩 H-S500TER-514500005W(i5-14500/16G/1TB SSD/W11)

Intel i5 ASUS PC 共 4 筆，總價 118654
平均價格 = 29663.5 元


### Task 4

In [5]:
import math
import time

PROD_API = ("https://ecapi.pchome.com.tw/ecshop/prodapi/v2/prod/"
            "{pid}&fields=Id,Price&_callback=jsonp_cb")
SRC_FILE = "products.txt"
OUTPUT_FILE = "standardization.csv"
SLEEP_SEC = 0.2
ZSCORE_NDIGITS = 6


def load_product_ids(path):
    """讀取 Task 1 產生的商品 ID（一行一個）。"""
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def get_price(pid):
    data = extract_json(fetch(PROD_API.format(pid=pid)))
    info = data.get(pid, {}) if isinstance(data, dict) else {}
    return (info.get("Price") or {}).get("P")


def population_mean_std(values):
    n = len(values)
    mean = sum(values) / n
    variance = sum((x - mean) ** 2 for x in values) / n
    return mean, math.sqrt(variance)


def main():
    ids = load_product_ids(SRC_FILE)

    rows = []
    for pid in ids:
        price = get_price(pid)
        if price is not None:
            rows.append((pid, price))
        time.sleep(SLEEP_SEC)

    prices = [p for _, p in rows]
    mean, std = population_mean_std(prices)
    print(f"母體 N={len(prices)}，平均={mean:.4f}，母體標準差={std:.4f}\n")

    with open(OUTPUT_FILE, "w", encoding="utf-8", newline="") as f:
        for pid, price in rows:
            z = (price - mean) / std
            line = f"{pid},{price},{round(z, ZSCORE_NDIGITS)}"
            f.write(line + "\n")
            print(line)

    print(f"\n完成！共 {len(rows)} 筆，已寫入 {OUTPUT_FILE}")


if __name__ == "__main__":
    main()


母體 N=46，平均=32944.3913，母體標準差=9242.3672

DSAA31-A900JQR50-000,22990,-1.077039
DSAA31-1900K0SXN-000,35990,0.329527
DSAA31-A900JQRLA-000,18990,-1.509829
DSAA31-A900JTIUG-000,22990,-1.077039
DSAA31-A900K19GS-000,22990,-1.077039
DSAA31-1900JVJRC-000,23990,-0.968842
DSAA31-A900K23IM-000,23990,-0.968842
DSAA31-A900JQRNZ-000,30990,-0.21146
DSAA31-A900JQR5L-000,24990,-0.860644
DSAA31-1900JZ4LO-000,33990,0.113132
DSAA31-1900JZ44W-000,29990,-0.319657
DSAA31-A900K1RRX-000,25990,-0.752447
DSAA31-A900JQR53-000,21990,-1.185237
DSAA31-A900K1VYC-000,29888,-0.330694
DSAA31-1900K1BBW-000,24888,-0.871681
DSAA31-A900IFZBC-000,25888,-0.763483
DSAAOC-1900JZ40C-000,24990,-0.860644
DSAA31-A900K282V-000,32888,-0.006101
DSAA31-A900K1RRB-000,25990,-0.752447
DSAA31-A900IFZEN-000,47990,1.627896
DSAA31-A900JQRNY-000,47990,1.627896
DSAA31-A900J29WS-000,43990,1.195106
DSAA31-A900J1950-000,43990,1.195106
DSAA31-A900IDS7J-000,43990,1.195106
DSAA31-A900JTLW5-000,43990,1.195106
DSAA31-1900JZ4TK-000,47990,1.627896
DSAA31-A9